# Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!git clone --quiet --recursive https://github.com/cvg/Hierarchical-Localization/
%cd Hierarchical-Localization
!pip install --progress-bar off --quiet -e .
!pip install --progress-bar off --quiet --upgrade plotly
# only build with ONNX gpu + caspar + ceres cuda/cudss; downloads disabled, so model paths must be set
!pip uninstall -y --quiet pycolmap
!pip install --progress-bar off --quiet \
   "https://github.com/lyehe/build_gpu_colmap/releases/download/v4.1.0/pycolmap-4.1.0+cu128.bundled.cudss-cp312-cp312-manylinux_2_35_x86_64.whl"

In [ ]:
from tqdm import tqdm
from pathlib import Path
import json
import numpy as np
import os
import shutil
import urllib.request


from hloc import reconstruction, visualization
from hloc.visualization import plot_images, read_image
from hloc.utils import viz_3d
import pycolmap

In [ ]:
DATA_PATH = next(p for p in [Path('/kaggle/input/datasets/yu5uf5/buggy-hloc'),
                             Path('/kaggle/input/buggy-hloc')] if p.exists())
outputs = Path('/kaggle/working/multi')
outputs.mkdir(exist_ok=True)
IMAGES_PATH = Path('/tmp/images')

In [ ]:
# camera position relative to the racebox in body frame, meters: {roll_id: (forward, left)}
# +forward = camera ahead of the racebox, +left = camera to its left
RB_CAM_OFFSET = {
    37: (1.25, 0), 38: (1.25, 0),
    39: (1.20, 0),
    44: (-0.80, 0), 45: (-0.80, 0),
    1387: (0.05, 0),
    1388: (-0.75, 0),
    1401: (-0.70, 0)
}

R_EARTH = 6371000.0

def apply_rb_cam_offset(gps, fwd, left):
    """Move racebox positions to the camera using gps-derived heading."""
    lat = np.array([s['lat'] for s in gps], float)
    lon = np.array([s['long'] for s in gps], float)
    latr = np.radians(lat)
    i = np.arange(len(gps))
    i0, i1 = np.maximum(i - 12, 0), np.minimum(i + 12, len(gps) - 1)  # ~0.5 s window at 25 Hz
    dn = np.radians(lat[i1] - lat[i0]) * R_EARTH
    de = np.radians(lon[i1] - lon[i0]) * R_EARTH * np.cos(latr)
    ok = np.hypot(de, dn) > 0.5
    if not ok.any():
        return
    head = np.arctan2(de, dn)
    last = np.maximum.accumulate(np.where(ok, i, -1))
    last[last < 0] = np.flatnonzero(ok)[0]  # hold heading through standstill
    head = head[last]
    off_e = fwd * np.sin(head) - left * np.cos(head)
    off_n = fwd * np.cos(head) + left * np.sin(head)
    for s, oe, on, phi in zip(gps, off_e, off_n, latr):
        s['lat'] += np.degrees(on / R_EARTH)
        s['long'] += np.degrees(oe / (R_EARTH * np.cos(phi)))

In [ ]:
import cv2
from collections import defaultdict
from itertools import combinations
from scipy.spatial import cKDTree


def load_vid_imu(vid_imu_path):
    """Load vid_imu exports; shift racebox onto the virb timeline and move it to the camera."""
    data = {}
    for p in sorted(vid_imu_path.glob('*.json')):
        with open(p) as f:
            data[p.stem] = json.load(f)
    for run, d in data.items():
        # racebox offset is imu-estimated in smooth.ipynb's export; virb streams need no shift
        off_ns = d.get('racebox_offset_ms', 0.0) * 1e6
        for key in ('racebox_gps', 'racebox_speed'):
            for s in d.get(key, []):
                s['timestamp'] += off_ns
        fwd, left = RB_CAM_OFFSET.get(int(run), (0.0, 0.0))
        if fwd or left:
            apply_rb_cam_offset(d.get('racebox_gps', []), fwd, left)
    return data


def sharpness_score(bgr):
    g = cv2.resize(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY), (426, 240))
    return float(cv2.Laplacian(g, cv2.CV_32F).var())


def extract_frames(video, start_ns, out_dir, stride=1):
    """Save frames as <ts_ns>.jpg plus sharpness.json; skips folders already done."""
    if (out_dir / 'sharpness.json').exists():
        return
    out_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video))
    if not cap.isOpened():
        raise RuntimeError(f"Could not open video: {video}")
    scores = {}
    i = 0
    try:
        with tqdm(total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), desc=out_dir.name, leave=False) as progress:
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                i += 1
                progress.update(1)
                if i % stride != 0:
                    continue
                ts_ns = start_ns + int(round(cap.get(cv2.CAP_PROP_POS_MSEC) * 1_000_000))
                cv2.imwrite(str(out_dir / f"{ts_ns}.jpg"), frame)
                scores[ts_ns] = sharpness_score(frame)
    finally:
        cap.release()
    with open(out_dir / 'sharpness.json', 'w') as f:
        json.dump(scores, f)
    print(f"{out_dir.name}: saved {len(scores)} frames")


def gps_enu(d, field):
    gps = d[field]
    ts, lat, lon, alt = (np.array([s[k] for s in gps], float)
                         for k in ('timestamp', 'lat', 'long', 'alt'))
    enu = np.array(gt.ellipsoid_to_enu(list(np.stack([lat, lon, alt], 1)), LAT0, LON0, ALT0))
    enu[:, 2] += CAM_HEIGHT
    return ts, enu


def speed_arrays(d, kind):
    if kind == 'racebox':
        spd = d['racebox_speed']
        return (np.array([s['timestamp'] for s in spd], float),
                np.array([s['speed'] for s in spd], float))
    vel = d['velocity']
    return (np.array([s['timestamp'] for s in vel], float),
            np.hypot([s['vx'] for s in vel], [s['vy'] for s in vel]))


def select_frames(run, d, gps_kind, delta_s):
    """Distance-uniform selection over the video ∩ gps window.
    Returns ({names, enu, heading}, (gps_ts, enu_src))."""
    image_paths = sorted((IMAGES_PATH / run).glob('*.jpg'), key=lambda p: int(p.stem))
    image_ts = np.array([int(p.stem) for p in image_paths])
    gps_ts, enu_src = gps_enu(d, 'racebox_gps' if gps_kind == 'racebox' else 'gps_data')
    spd_ts, spd = speed_arrays(d, gps_kind)

    m = (spd_ts >= max(gps_ts[0], image_ts[0])) & (spd_ts <= min(gps_ts[-1], image_ts[-1]))
    ts_w, v_w = spd_ts[m], spd[m]
    dist = np.concatenate([[0.0], np.cumsum(np.diff(ts_w) / 1e9 * (v_w[1:] + v_w[:-1]) / 2)])
    sel_ts = np.interp(np.arange(0.0, dist[-1], delta_s), dist, ts_w)

    # sharpest frame within each target's window (blur is vibration-driven, varies frame to frame)
    scores = {}
    score_file = IMAGES_PATH / run / 'sharpness.json'
    if score_file.exists():
        with open(score_file) as f:
            scores = {int(k): v for k, v in json.load(f).items()}
    gaps = np.diff(sel_ts)
    bounds = np.concatenate([[sel_ts[0] - gaps[0] / 2],
                             (sel_ts[:-1] + sel_ts[1:]) / 2,
                             [sel_ts[-1] + gaps[-1] / 2]])
    idx = []
    for k, t in enumerate(sel_ts):
        i0, i1 = np.searchsorted(image_ts, (bounds[k], bounds[k + 1]))
        if i0 >= i1:
            idx.append(np.abs(image_ts - t).argmin())
        elif scores:
            idx.append(i0 + int(np.argmax([scores.get(int(u), 0.0) for u in image_ts[i0:i1]])))
        else:
            idx.append(i0 + np.abs(image_ts[i0:i1] - t).argmin())
    idx = np.unique(idx)

    ts = image_ts[idx]
    enu = np.stack([np.interp(ts, gps_ts, enu_src[:, i]) for i in range(3)], 1)
    grad = np.gradient(enu[:, :2], axis=0)
    print(f'{run}: {len(idx)} frames over {dist[-1]:.0f}m')
    return ({'names': [str(image_paths[i].relative_to(IMAGES_PATH)) for i in idx],
             'enu': enu,
             'heading': np.arctan2(grad[:, 1], grad[:, 0])},
            (gps_ts, enu_src))


def sequential_pairs(names, k):
    return {(names[i], names[j])
            for i in range(len(names))
            for j in range(i + 1, min(i + 1 + k, len(names)))}


def cross_pairs(sel_a, sel_b, k, r, heading_max_deg):
    """Proximity pairs between two selections, gated on heading difference."""
    tree = cKDTree(sel_b['enu'][:, :2])
    dists, nbrs = tree.query(sel_a['enu'][:, :2], k=k, distance_upper_bound=r)
    if k == 1:
        dists, nbrs = dists[:, None], nbrs[:, None]
    pairs = set()
    for i, (ds, js) in enumerate(zip(dists, nbrs)):
        for dist, j in zip(ds, js):
            if not np.isfinite(dist):
                continue
            dh = abs(sel_a['heading'][i] - sel_b['heading'][j])
            if min(dh, 2 * np.pi - dh) <= np.radians(heading_max_deg):
                pairs.add(tuple(sorted((sel_a['names'][i], sel_b['names'][j]))))
    return pairs

In [ ]:
MODELS_PATH = Path('/tmp/models')
MASKS_PATH = Path('/tmp/masks')


def stage_models():
    # this pycolmap build can't auto-download onnx models; stage them locally
    MODELS_PATH.mkdir(exist_ok=True)
    for name in ('aliked-n16rot.onnx', 'aliked-lightglue.onnx'):
        f = MODELS_PATH / name
        if not f.exists():
            urllib.request.urlretrieve(
                f'https://github.com/colmap/colmap/releases/download/3.13.0/{name}', f)


def link_masks(names):
    # colmap masks: <mask_path>/<image name>.png, black = exclude; all frames share mask0
    MASKS_PATH.mkdir(exist_ok=True)
    mask_src = MASKS_PATH / 'mask0.png'
    if not mask_src.exists():
        shutil.copy(DATA_PATH / 'mask0.png', mask_src)
    for ref in names:
        dst = MASKS_PATH / f'{ref}.png'
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            os.link(mask_src, dst)


def extract_image_features(db_path, names):
    options = pycolmap.FeatureExtractionOptions()
    options.type = pycolmap.FeatureExtractorType.ALIKED_N16ROT
    options.aliked.n16rot_model_path = str(MODELS_PATH / 'aliked-n16rot.onnx')
    options.aliked.max_num_features = 4096
    options.use_gpu = True
    options.gpu_index = GPU_INDEX
    pycolmap.extract_features(
        db_path, IMAGES_PATH, image_names=sorted(names),
        camera_mode=pycolmap.CameraMode.PER_FOLDER,  # consider PER_IMAGE because of stabilization
        reader_options={'camera_model': 'SIMPLE_RADIAL',
                        'camera_params': ','.join(str(v) for v in CAMERA_PARAMS),
                        'mask_path': str(MASKS_PATH)},
        extraction_options=options)


def match_pairs(db_path, pairs_file):
    matching_options = pycolmap.FeatureMatchingOptions()
    matching_options.type = pycolmap.FeatureMatcherType.ALIKED_LIGHTGLUE
    matching_options.aliked.lightglue.model_path = str(MODELS_PATH / 'aliked-lightglue.onnx')
    matching_options.use_gpu = True
    matching_options.gpu_index = GPU_INDEX
    pairing_options = pycolmap.ImportedPairingOptions()
    pairing_options.match_list_path = str(pairs_file)
    pycolmap.match_image_pairs(db_path, matching_options=matching_options,
                               pairing_options=pairing_options)


def write_pose_priors(db_path, gps_by_run, cov):
    """Position priors for images of runs in gps_by_run; inert unless use_prior_position."""
    with pycolmap.Database.open(str(db_path)) as db:
        have = {p.corr_data_id.id for p in db.read_all_pose_priors()}
        for image in db.read_all_images():
            p = Path(image.name)
            if image.data_id.id in have or p.parts[0] not in gps_by_run:
                continue
            gps_ts, enu = gps_by_run[p.parts[0]]
            ts = int(p.stem)
            prior = pycolmap.PosePrior()
            prior.corr_data_id = image.data_id
            prior.position = np.array([np.interp(ts, gps_ts, enu[:, i]) for i in range(3)])
            prior.position_covariance = cov
            prior.coordinate_system = pycolmap.PosePriorCoordinateSystem.CARTESIAN
            db.write_pose_prior(prior)
        print('pose priors:', db.num_pose_priors())

In [ ]:
videos = {v.stem: v for v in DATA_PATH.glob('*.mp4')}
data = load_vid_imu(DATA_PATH / 'vid_imu')

# Create Images

In [ ]:
for stem, video in tqdm(videos.items(), desc='videos'):
    extract_frames(video, data[stem]['camera_start'], IMAGES_PATH / stem)

# Config

In [ ]:
sfm_pairs = outputs / 'pairs-sfm.txt'
sfm_dir = outputs / 'sfm'
sfm_prior_dir = outputs / 'sfm_prior'
database = outputs / 'database.db'

RESUME = False     # continue from outputs/database.db + outputs/prior
RECON_RUNS = None  # runs to reconstruct this pass; None = all

In [ ]:
# racebox: 25Hz + scalar doppler speed; fit: 10Hz + velocity vectors
GPS_KIND = 'racebox'
LAT0, LON0, ALT0 = 40.44163016, -79.94165829, 288.42151354  # shared ENU reference
gt = pycolmap.GPSTransform(pycolmap.GPSTransfromEllipsoid.WGS84)

DELTA_S = 2.0         # m between selected frames
SEQ_K = 6             # forward sequential pairs per frame
CROSS_K = 3           # nearest cross-run candidates per frame
CROSS_R = 6.0         # m, max cross-run pair distance
HEADING_MAX_DEG = 40  # max cross-run heading difference

CAM_HEIGHT = 0.35  # m, gps alt is DEM road level; lift priors approximatley to the camera

In [ ]:
import torch

GPU_INDEX = ','.join(str(i) for i in range(torch.cuda.device_count()))

# caspar BA only supports SIMPLE_RADIAL [f, cx, cy, k1]/PINHOLE
# initial estimates refined per image
CAMERA_PARAMS = [653.4, 631.72, 338.74, -0.0526]

# Mapping

## Select Frames

In [ ]:
runs = sorted(data, key=int)
sel, run_enu = {}, {}
for run in runs:
    sel[run], run_enu[run] = select_frames(run, data[run], GPS_KIND, DELTA_S)

# selection must reproduce db names (guards param drift across sessions)
if RESUME:
    with pycolmap.Database.open(str(database)) as db:
        db_names = {im.name for im in db.read_all_images()}
    for run in runs:
        mine = {n for n in db_names if n.startswith(f'{run}/')}
        assert not mine or mine == set(sel[run]['names']), run

references = [n for run in runs for n in sel[run]['names']]
recon_names = sorted(n for r in (RECON_RUNS or runs) for n in sel[r]['names'])
len(references)

In [ ]:
import matplotlib.pyplot as plt

for run in runs:
    plt.plot(*sel[run]['enu'][:, :2].T, '.', ms=2, label=run)
plt.axis('equal')
plt.legend(markerscale=5)
plt.show()

In [ ]:
sl = slice(200, 210)
plot_images([read_image(IMAGES_PATH / ref) for ref in references[sl]],
            titles=references[sl], dpi=25)

## Features

In [ ]:
stage_models()
link_masks(references)

In [ ]:
pycolmap.logging.set_log_destination(pycolmap.logging.INFO, outputs / 'colmap.LOG.')
if not RESUME:
    database.unlink(missing_ok=True)
extract_image_features(database, references)

## Matching

In [ ]:
pairs = set()
for run in runs:
    pairs |= sequential_pairs(sel[run]['names'], SEQ_K)
n_seq = len(pairs)

for a, b in combinations(runs, 2):
    pairs |= cross_pairs(sel[a], sel[b], CROSS_K, CROSS_R, HEADING_MAX_DEG)

sfm_pairs.write_text('\n'.join(f'{a} {b}' for a, b in sorted(pairs)))
print(f'{n_seq} sequential + {len(pairs) - n_seq} cross-run pairs')

In [ ]:
match_pairs(database, sfm_pairs)

## Reconstruct

### Database

In [ ]:
PRIOR_STD_XY = 0.5
PRIOR_STD_Z = 1.0
# inert unless use_prior_position is set, so both reconstructions can share the db
write_pose_priors(database, run_enu,
                  np.diag([PRIOR_STD_XY**2, PRIOR_STD_XY**2, PRIOR_STD_Z**2]))

### No Prior

In [ ]:
# model = reconstruction.run_reconstruction(
#     sfm_dir, database, IMAGES_PATH, verbose=True,
#     options={
#         "image_names": recon_names,
#         "ba_local_backend": pycolmap.BundleAdjustmentBackend.CASPAR,
#         "ba_global_backend": pycolmap.BundleAdjustmentBackend.CASPAR,
#     })

In [ ]:
# p = outputs / 'no_prior'
# p.mkdir(parents=True, exist_ok=True)
# model.write(p)

### Prior

In [ ]:
# priors only constrain global BA, and caspar doesn't support them
prior_options = {
    "image_names": recon_names,
    "use_prior_position": True,
    # "use_robust_loss_on_prior_position": True,
    # Speed options
    # "ba_use_gpu": True,
    # "ba_local_backend": pycolmap.BundleAdjustmentBackend.CASPAR,
    # "ba_global_backend": pycolmap.BundleAdjustmentBackend.CERES,
    # "ba_global_frames_ratio": 1.3,
    # "ba_global_points_ratio": 1.3,
    # "ba_local_max_num_iterations": 12,
    # "ba_local_max_refinements": 2,
    # "ba_global_max_num_iterations": 30,
    # "ba_global_max_refinements": 3,
    # "mapper": {"ba_global_ignore_redundant_points3D": True},
}
if RESUME:
    shutil.rmtree(sfm_prior_dir, ignore_errors=True)
    sfm_prior_dir.mkdir(parents=True)
    with pycolmap.ostream():
        recs = pycolmap.incremental_mapping(
            database, IMAGES_PATH, sfm_prior_dir,
            options={**prior_options, "fix_existing_frames": False},
            input_path=str(outputs / 'prior'))
    model_prior = recs[0]
else:
    model_prior = reconstruction.run_reconstruction(
        sfm_prior_dir, database, IMAGES_PATH, verbose=True, options=prior_options)

In [ ]:
p = outputs / 'prior'
p.mkdir(parents=True, exist_ok=True)
model_prior.write(p)

# Visualize

### No Prior

In [13]:
# model = pycolmap.Reconstruction(str(DATA_PATH / 'outputs' / 'no_prior'))

In [15]:
# fig = viz_3d.init_figure()
# viz_3d.plot_reconstruction(fig, model, points_rgb=True)
# fig.show()

### Prior

In [ ]:
model_prior = pycolmap.Reconstruction(str(outputs / 'prior'))

In [ ]:
with pycolmap.Database.open(str(database)) as db:
    priors = db.read_all_pose_priors()
enu = {p.corr_data_id.id: p.position for p in priors}  # already ENU
errs = np.array([model_prior.images[i].projection_center() - enu[i]
                 for i in model_prior.reg_image_ids()])
print(f"rmse vs gps: {np.sqrt((errs[:, :2] ** 2).sum(1).mean()):.2f} m horizontal, "
      f"{np.sqrt((errs[:, 2] ** 2).mean()):.2f} m vertical")

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "notebook_connected"

by_run = {}
for i in sorted(model_prior.reg_image_ids()):
    by_run.setdefault(model_prior.images[i].name.split('/')[0], []).append(i)

fig = go.Figure()
colors = px.colors.qualitative.Plotly
for k, run in enumerate(sorted(by_run, key=int)):
    ids = by_run[run]
    pri = np.array([enu[i] for i in ids])
    sol = np.array([model_prior.images[i].projection_center() for i in ids])
    names = [model_prior.images[i].name for i in ids]
    c = colors[k % len(colors)]
    seg = np.concatenate([pri[:, None, :2], sol[:, None, :2],
                          np.full((len(ids), 1, 2), np.nan)], 1).reshape(-1, 2)
    fig.add_scatter(x=seg[:, 0], y=seg[:, 1], mode='lines', hoverinfo='skip',
                    line=dict(color='lightgray', width=1),
                    legendgroup=run, showlegend=False)
    fig.add_scatter(x=pri[:, 0], y=pri[:, 1], mode='markers', text=names,
                    marker=dict(color=c, size=4),
                    name=f'{run} prior', legendgroup=run)
    fig.add_scatter(x=sol[:, 0], y=sol[:, 1], mode='markers', text=names,
                    marker=dict(color=c, size=5, symbol='x'),
                    name=f'{run} solved', legendgroup=run)
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.update_layout(height=700)
fig.show()

In [ ]:
fig = viz_3d.init_figure()
viz_3d.plot_reconstruction(fig, model_prior, points_rgb=True)
fig.show()

# Localization

Register query runs against the fixed `prior_all` model, one run at a time. Runs on copies — the original database/model are never modified — and query frames only match the map (no pairs between localized runs). Runnable standalone: Setup → Config → here.

In [ ]:
map_database = DATA_PATH / 'database.db'
map_model_path = DATA_PATH / 'prior_all'
loc_work = Path('/tmp/loc')  # db copies + scratch
loc_out = Path('/kaggle/working/loc')

LOC_RUNS = None        # subset of int run ids; None = all of test_vid_imu
LOC_DELTA_S = 2.0      # m between query frames
LOC_GPS = 'racebox'    # 'racebox' | 'virb': selection + pairing source (virb skips uncovered runs)
LOC_SEQ_K = SEQ_K
LOC_CROSS_K = CROSS_K  # map candidates per query frame, per map run
LOC_CROSS_R = CROSS_R
LOC_PRIORS = False     # virb position priors on query frames
LOC_PRIOR_STD_XY, LOC_PRIOR_STD_Z = 3.0, 5.0

In [ ]:
loc_videos = {v.stem: v for v in (DATA_PATH / 'test_vid').glob('*.[mM][pP]4')}
loc_data = load_vid_imu(DATA_PATH / 'test_vid_imu')
loc_runs = [r for r in sorted(loc_data, key=int)
            if LOC_RUNS is None or int(r) in LOC_RUNS]
if LOC_GPS == 'virb':
    skipped = [r for r in loc_runs if not loc_data[r]['gps_data']]
    loc_runs = [r for r in loc_runs if loc_data[r]['gps_data']]
    if skipped:
        print('no virb gps, skipping:', skipped)

for run in tqdm(loc_runs, desc='videos'):
    extract_frames(loc_videos[run], loc_data[run]['camera_start'], IMAGES_PATH / run)
loc_runs

In [ ]:
loc_sel = {}
for run in loc_runs:
    loc_sel[run], _ = select_frames(run, loc_data[run], LOC_GPS, LOC_DELTA_S)

# map frame positions/headings come from the model itself, not the mapping vid_imu
map_model = pycolmap.Reconstruction(str(map_model_path))
by_run = defaultdict(list)
for i in map_model.reg_image_ids():
    im = map_model.images[i]
    p = Path(im.name)
    by_run[p.parts[0]].append((int(p.stem), im.name, im.projection_center()))
map_sel = {}
for run, items in by_run.items():
    items.sort()
    enu = np.array([c for _, _, c in items])
    grad = np.gradient(enu[:, :2], axis=0)
    map_sel[run] = {'names': [n for _, n, _ in items], 'enu': enu,
                    'heading': np.arctan2(grad[:, 1], grad[:, 0])}
print({run: len(s['names']) for run, s in map_sel.items()})

In [ ]:
stage_models()
link_masks([n for run in loc_runs for n in loc_sel[run]['names']])
loc_work.mkdir(parents=True, exist_ok=True)
local_map_db = loc_work / 'map_database.db'
if not local_map_db.exists():
    shutil.copy(map_database, local_map_db)  # Drive reads are slow; stage the 4.4 GB db once

In [ ]:
pycolmap.logging.set_log_destination(pycolmap.logging.INFO, loc_work / 'colmap.LOG.')
loc_models = {}
for run in loc_runs:
    names = loc_sel[run]['names']
    work = loc_work / run
    db = work / 'database.db'
    if db.exists():  # reuse features/matches only if the selection is unchanged
        with pycolmap.Database.open(str(db)) as dbh:
            same = {im.name for im in dbh.read_all_images()
                    if im.name.split('/')[0] == run} == set(names)
        if not same:
            shutil.rmtree(work)
    if not db.exists():
        work.mkdir(parents=True, exist_ok=True)
        shutil.copy(local_map_db, db)
        extract_image_features(db, names)

    pairs = sequential_pairs(names, LOC_SEQ_K)
    for m in map_sel.values():
        pairs |= cross_pairs(loc_sel[run], m, LOC_CROSS_K, LOC_CROSS_R, HEADING_MAX_DEG)
    (work / 'pairs.txt').write_text('\n'.join(f'{a} {b}' for a, b in sorted(pairs)))
    match_pairs(db, work / 'pairs.txt')

    if LOC_PRIORS:
        write_pose_priors(db, {run: gps_enu(loc_data[run], 'gps_data')},
                          np.diag([LOC_PRIOR_STD_XY**2, LOC_PRIOR_STD_XY**2, LOC_PRIOR_STD_Z**2]))

    # priors only constrain global BA, and caspar doesn't support them
    opt = {'fix_existing_frames': True, 'use_prior_position': LOC_PRIORS,
           'ba_use_gpu': True,
           'ba_local_backend': pycolmap.BundleAdjustmentBackend.CASPAR,
           'ba_global_backend': (pycolmap.BundleAdjustmentBackend.CERES if LOC_PRIORS
                                 else pycolmap.BundleAdjustmentBackend.CASPAR)}

    shutil.rmtree(work / 'model', ignore_errors=True)
    (work / 'model').mkdir()
    with pycolmap.ostream():
        recs = pycolmap.incremental_mapping(db, IMAGES_PATH, work / 'model',
                                            options=opt, input_path=str(map_model_path))
    loc_models[run] = recs[0]
    reg = {loc_models[run].images[i].name for i in loc_models[run].reg_image_ids()}
    print(f'{run}: registered {len(reg & set(names))}/{len(names)} query frames')

In [ ]:
loc_results = {}
for run, model in loc_models.items():
    names = set(loc_sel[run]['names'])
    rb_ts, rb_enu = gps_enu(loc_data[run], 'racebox_gps')
    frames = []
    for i in sorted(model.reg_image_ids()):
        im = model.images[i]
        if im.name not in names:
            continue
        ts = int(Path(im.name).stem)
        c = im.projection_center()
        rb = np.array([np.interp(ts, rb_ts, rb_enu[:, j]) for j in range(3)])
        frames.append({'ts': ts, 'enu': c.tolist(),
                       'quat_xyzw': im.cam_from_world().rotation.quat.tolist(),
                       'racebox_err': (c - rb).tolist()})
    loc_results[run] = frames

    out = loc_out / run
    out.mkdir(parents=True, exist_ok=True)
    model.write(out)
    with open(out / 'poses.json', 'w') as f:
        json.dump({'delta_s': LOC_DELTA_S, 'gps': LOC_GPS, 'priors': LOC_PRIORS,
                   'frames': frames}, f)

    errs = np.array([f['racebox_err'] for f in frames])
    h = np.hypot(errs[:, 0], errs[:, 1])
    print(f"{run}: {len(frames)}/{len(names)} frames, vs racebox "
          f"{np.sqrt((h**2).mean()):.2f}m rmse / {np.percentile(h, 90):.2f}m p90 horizontal, "
          f"{np.sqrt((errs[:, 2]**2).mean()):.2f}m rmse vertical")

In [ ]:
# reload saved localizations (skip the reconstruction cells above)
# import json
# from pathlib import Path
# loc_out = Path('../../../.././tmp/loc_out')
loc_models, loc_results = {}, {}
for p in sorted(loc_out.iterdir()):
    if (p / 'poses.json').exists():
        with open(p / 'poses.json') as f:
            loc_results[p.name] = json.load(f)['frames']
        loc_models[p.name] = pycolmap.Reconstruction(str(p))
print(sorted(loc_results))

['39']


In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
import numpy as np
# pio.renderers.default = "notebook_connected"

fig = go.Figure()
colors = px.colors.qualitative.Plotly
for k, run in enumerate(sorted(loc_results, key=int)):
    frames = loc_results[run]
    enu = np.array([f['enu'] for f in frames])
    errs = np.array([f['racebox_err'] for f in frames])
    rb = enu - errs
    text = [f'{run}/{f["ts"]}: {np.hypot(*e[:2]):.2f}m' for f, e in zip(frames, errs)]
    c = colors[k % len(colors)]
    seg = np.concatenate([rb[:, None, :2], enu[:, None, :2],
                          np.full((len(frames), 1, 2), np.nan)], 1).reshape(-1, 2)
    fig.add_scatter(x=seg[:, 0], y=seg[:, 1], mode='lines', hoverinfo='skip',
                    line=dict(color='lightgray', width=1),
                    legendgroup=run, showlegend=False)
    fig.add_scatter(x=rb[:, 0], y=rb[:, 1], mode='markers', text=text,
                    marker=dict(color=c, size=4),
                    name=f'{run} racebox', legendgroup=run)
    fig.add_scatter(x=enu[:, 0], y=enu[:, 1], mode='markers', text=text,
                    marker=dict(color=c, size=5, symbol='x'),
                    name=f'{run} loc', legendgroup=run)
fig.update_yaxes(scaleanchor='x', scaleratio=1)
fig.update_layout(height=700)
fig.show()

fig = go.Figure()
for k, run in enumerate(sorted(loc_results, key=int)):
    frames = sorted(loc_results[run], key=lambda f: f['ts'])
    ts = np.array([f['ts'] for f in frames]) / 1e9
    h = np.array([np.hypot(*f['racebox_err'][:2]) for f in frames])
    fig.add_scatter(x=ts - ts[0], y=h, mode='markers', text=[f['ts'] for f in frames],
                    marker=dict(color=colors[k % len(colors)], size=4), name=run)
fig.update_layout(height=350, xaxis_title='s', yaxis_title='horizontal error m')
fig.show()